# The full differential geometry toolkit on an arbitrary 2D surface

We consolidate all the geometric notions introduced in this notebook into a single, comprehensive symbolic and numerical exploration on a 2D manifold. While the code below is initialized with the hyperbolic plane in geodesic polar coordinates ($ds^2 = du^2 + \cosh^2(u)\,dv^2$) as a working example, **the entire pipeline is completely metric-agnostic**. You can replace the metric tensor $g_{ij}$ in Part 0 with *any* 2D Riemannian metric, and the toolkit will automatically derive and verify all downstream geometric structures.

We explicitly construct and verify:
1. **Tangent & Cotangent Bundles (Musical Isomorphisms)**: The metric $g$ and its inverse $g^{-1}$ map vector fields (tangent bundle) to 1-forms (cotangent bundle) via the flat ($\flat$) and sharp ($\sharp$) operators.
2. **Levi-Civita Connection**: The Christoffel symbols $\Gamma^k_{ij}$, encoding how basis vectors change. We verify torsion-freeness ($\Gamma^k_{ij} = \Gamma^k_{ji}$) and metric compatibility ($\nabla_k g_{ij} = 0$).
3. **Curvature**: The Riemann tensor $R^\rho_{\sigma\mu\nu}$, its symmetries, the Ricci tensor $R_{\mu\nu}$, and the scalar curvature $R$. We extract the Gauss curvature $K = R/2$.
4. **Differential Forms & Operators**: The exterior derivative $d$, the Hodge star $\star$, the codifferential $\delta$, and the Laplace-Beltrami operator $\Delta = d\delta + \delta d$. We verify $\star^2 = -1$ on 1-forms and that $\Delta$ matches the coordinate formula.
5. **Geodesics & Holonomy**: The geodesic equation $\ddot{x}^k + \Gamma^k_{ij}\dot{x}^i\dot{x}^j = 0$, Jacobi fields (geodesic deviation), and holonomy around closed loops.

This serves as a complete "dictionary" translating abstract differential geometry into the concrete symbolic and numerical machinery of `psiop` and `riemannian`.

In [ ]:
"""
Full differential geometry for a given 2D metric.

Explores:
  - Operators, vectors, forms, and their interactions
  - Connection, vectors, forms, and their interactions
  - Hodge theory, Laplacians, parallel transport, geodesics, Jacobi fields
  - Numerical Hodge decomposition with visualization
"""

from psiop import * 
from riemannian import *

plt.rcParams.update({'figure.dpi': 100, 'font.size': 9})

# ===========================
# PART 0: SETUP — The Metric
# ===========================
print("=" * 80)
print("PART 0: THE METRIC")
print("=" * 80)

u, v = symbols('u v', real=True)
coords = (u, v)

# Use the parametrization of a surface and convet it into a metric
# Standard Mobius strip parametrization:
#   u in [0, 2*pi)  — goes around the loop
#   v in [-1, 1]     — across the width of the strip
s1 = (1 + v/2*cos(u/2)) * cos(u)
s2 = (1 + v/2*cos(u/2)) * sin(u)
s3 = v/2 * sin(u/2)

g_S = surface2metric((s1, s2, s3), (u, v))
print(g_S)

# Or use a metric directly
R, a = 2, 1   # major / minor radius
g_torus = sp.Matrix([[a**2, 0],
                     [0, (R + a*sp.cos(u))**2]])

g_sphere = Matrix([[1, 0], [0, sin(u)**2]])

g_hypbplan = Matrix([[1, 0], [0, cosh(u)**2]])

g_paraboloid = sp.Matrix([[1 + u**2, 0],
                          [0, u**2]])

g_warp = sp.Matrix([[1, 0],
                    [0, (1 + u**2)**2]])

g_flat = sp.Matrix([[1, 0],
                    [0, 1]])

g_matrix = g_sphere
m = Metric(g_matrix, coords)
g_inv = g_matrix.inv()
det_g = g_matrix.det()
sqrt_g = sqrt(det_g) 

print(f"\n  Metric g_ij:")
sp.pprint(g_matrix)
print(f"\n  Inverse metric g^ij:")
sp.pprint(g_inv)
print(f"  det(g) = {det_g}")
print(f"  √|g|   = {sqrt_g}")

# ============================================================================
# PART 1: MUSICAL ISOMORPHISMS — Vectors ↔ 1-forms (♭ and ♯)
# ============================================================================
print("\n" + "=" * 80)
print("PART 1: MUSICAL ISOMORPHISMS  (♭: TM → T*M,  ♯: T*M → TM)")
print("=" * 80)

V1, V2 = symbols('V1 V2', real=True)
V = Matrix([V1, V2])

# ♭ (flat): V^i → V_i = g_ij V^j
V_flat = g_matrix * V
# ♯ (sharp): V_i → V^i = g^{ij} V_j
V_sharp = g_inv * V_flat

print(f"\n  V = ({V1}, {V2})  [contravariant vector]")
print(f"  V♭ = g·V = ({V_flat[0]}, {V_flat[1]})  [covariant 1-form]")
print(f"  (V♭)♯ = g⁻¹·V♭ = ({V_sharp[0]}, {V_sharp[1]})")
print(f"  Round-trip (V♭)♯ == V: {sp.simplify(V_sharp - V) == Matrix([0, 0])}")

# --- Visual: how ♭ distorts a vector field ---
print("\n  [VISUAL] Vector field V=(cos v, sin u) and its flat image V♭...")
u_vals = np.linspace(-2, 2, 20)
v_vals = np.linspace(0, 2*np.pi, 20)
U, V_grid = np.meshgrid(u_vals, v_vals, indexing='ij')

# Vector field components
Vx_field = np.cos(V_grid)
Vy_field = np.sin(U)

# Flat: V_flat_u = g_uu * V^u = 1 * V^u,  V_flat_v = g_vv * V^v = cosh²(u) * V^v
Vflat_u = Vx_field
Vflat_v = np.cosh(U)**2 * Vy_field

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (fx, fy, title) in zip(axes, [
    (Vx_field, Vy_field, r'Vector field $V = (\cos v,\, \sin u)$'),
    (Vflat_u, Vflat_v, r'Flat image $V^\flat = g_{ij} V^j$')
]):
    mag = np.sqrt(fx**2 + fy**2)
    ax.quiver(U, V_grid, fx, fy, mag, cmap='viridis', alpha=0.7, scale=25)
    ax.set_title(title)
    ax.set_xlabel('u'); ax.set_ylabel('v')
    ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('part1_musical_isomorphisms.png', dpi=150)
plt.show()
print("  → Saved: part1_musical_isomorphisms.png")

# ============================================================================
# PART 2: LEVI-CIVITA CONNECTION — How vectors change along curves
# ============================================================================
print("\n" + "=" * 80)
print("PART 2: LEVI-CIVITA CONNECTION  ∇")
print("=" * 80)

# 1. Christoffel symbols (Already computed and cached in m.christoffel_sym)
Gamma = m.christoffel_sym
print("\n  Non-zero Christoffel symbols Γ^k_ij:")
for i in range(2):
    for j in range(2):
        for k in range(2):
            if Gamma[i][j][k] != 0:
                print(f"    Γ^{coords[i]}_{coords[j]}{coords[k]} = {Gamma[i][j][k]}")

# (Metric compatibility ∇g = 0 is mathematically guaranteed by the Levi-Civita 
# definition used inside the Metric class, so we can skip the manual check).

# 2. Covariant derivative of a vector field
Vu, Vv = sin(v), cos(u)
nabla_V = m.covariant_derivative_vector([Vu, Vv])
print(f"\n  Covariant derivative ∇_i V^j:")
print(f"    ∇_u V = ({nabla_V[0,0]}, {nabla_V[0,1]})")
print(f"    ∇_v V = ({nabla_V[1,0]}, {nabla_V[1,1]})")

# 3. Covariant derivative of a 1-form
omega_u, omega_v = cos(u), sin(v)
nabla_omega = m.covariant_derivative_covector([omega_u, omega_v])
print(f"\n  Covariant derivative ∇_i ω_j:")
print(f"    ∇_u ω = ({nabla_omega[0,0]}, {nabla_omega[0,1]})")
print(f"    ∇_v ω = ({nabla_omega[1,0]}, {nabla_omega[1,1]})")

# --- Parallel transport along a geodesic (VISUAL) ---
print("\n  [VISUAL] Parallel transport of a vector along a geodesic...")
p0 = (0.5, 0.0)
v0 = (0.3, 1.0)
traj = geodesic_solver(m, p0, v0, (0, 4), method='rk4', n_steps=200)

# Transport vector (1, 0) along the geodesic
pt = parallel_transport(m, traj, (1.0, 0.0))

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
# Plot geodesic
ax.plot(traj['x'], traj['y'], 'b-', linewidth=2, label='Geodesic')
# Plot transported vector at intervals
step = 20
for i in range(0, len(pt['t']), step):
    x_pos, y_pos = traj['x'][i], traj['y'][i]
    vx, vy = pt['vx'][i], pt['vy'][i]
    # Scale for visibility
    scale = 0.3
    ax.arrow(x_pos, y_pos, scale*vx, scale*vy,
             head_width=0.05, head_length=0.03, fc='red', ec='red', alpha=0.7)
ax.plot(traj['x'][0], traj['y'][0], 'go', markersize=10, label='Start')
ax.plot(traj['x'][-1], traj['y'][-1], 'rs', markersize=10, label='End')
ax.set_xlabel('u'); ax.set_ylabel('v')
ax.set_title('Parallel Transport\n'
             r'(vector preserves $\langle V, V \rangle_g$ along the geodesic)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('part2_parallel_transport.png', dpi=150)
plt.show()
print("  → Saved: part2_parallel_transport.png")

# Verify norm preservation
norm_start = (m.g_func[(0,0)](traj['x'][0], traj['y'][0]) * pt['vx'][0]**2 +
              m.g_func[(1,1)](traj['x'][0], traj['y'][0]) * pt['vy'][0]**2)
norm_end = (m.g_func[(0,0)](traj['x'][-1], traj['y'][-1]) * pt['vx'][-1]**2 +
            m.g_func[(1,1)](traj['x'][-1], traj['y'][-1]) * pt['vy'][-1]**2)
print(f"  Norm preservation: |V|²_start = {norm_start:.6f}, |V|²_end = {norm_end:.6f}")

# ============================================================================
# PART 3: CURVATURE — Riemann, Ricci, Gauss
# ============================================================================
print("\n" + "=" * 80)
print("PART 3: CURVATURE TENSORS")
print("=" * 80)

# 1. Riemann tensor R^i_{jkl} (Returns a nested dict)
R_dict = m.riemann_tensor()

# Lower the first index to get fully covariant R_{ijkl} = g_{im} R^m_{jkl}
R_down = m.riemann_tensor_lower()

diff_pair = R_down[0][1][0][1] - R_down[1][0][1][0]
diff_anti = R_down[0][1][0][1] + R_down[1][0][0][1]

def is_zero(expr, subs_pt=None):
    e = simplify(expand_trig(expr))
    if e == 0:
        return True
    # numeric fallback: sample a point and check it's ~0
    pt = subs_pt or {u: 0.37, v: 1.1}
    return abs(complex(e.subs(pt))) < 1e-9

symm_check  = is_zero(diff_pair)
anti_check1 = is_zero(diff_anti)
print(f"\n  Riemann tensor symmetries verified: Pair={symm_check}, Antisym={anti_check1}")

# 2. Ricci Tensor, Scalar Curvature, and Gaussian Curvature
Ricci = m.ricci_tensor()
R_scalar = m.ricci_scalar()
K_scalar = m.gauss_curvature()  # Automatically computes R_1212 / det(g)

print(f"\n  Scalar curvature R = {R_scalar}")
print(f"  Gaussian curvature K = {K_scalar}")
print(f"  Ricci Tensor R_ij = K·g_ij:")
sp.pprint(Ricci)

# --- Visual: curvature map (DYNAMIC) ---
print("\n  [VISUAL] Gaussian curvature map...")
fig, ax = plt.subplots(figsize=(8, 6))
u_range = np.linspace(-3, 3, 100)
v_range = np.linspace(0, 2*np.pi, 100)
U_map, V_map = np.meshgrid(u_range, v_range, indexing='ij')

# 1. Clean up the symbolic expression and convert to a fast NumPy function
# (trigsimp ensures SymPy reduces complex trig fractions to their simplest form)
K_expr = sp.trigsimp(K_scalar) 
K_func = sp.lambdify((u, v), K_expr, 'numpy')

# 2. Evaluate the function on the meshgrid
K_vals = K_func(U_map, V_map)

# 3. Fallback for constant curvature: 
# If K is a constant (like -1 or 1), lambdify returns a single scalar float 
# instead of a 2D array. pcolormesh requires a 2D array, so we broadcast it.
if np.isscalar(K_vals) or np.shape(K_vals) != np.shape(U_map):
    K_vals = np.full_like(U_map, float(K_vals), dtype=float)

# 4. Dynamic color limits
vmin, vmax = np.min(K_vals), np.max(K_vals)
if vmin == vmax:
    # If perfectly constant, expand limits slightly so pcolormesh doesn't throw a warning
    vmin, vmax = vmin - 0.5, vmax + 0.5 

im = ax.pcolormesh(U_map, V_map, K_vals, cmap='RdBu_r', vmin=vmin, vmax=vmax)
plt.colorbar(im, ax=ax, label='Gaussian Curvature K')
ax.set_xlabel('u'); ax.set_ylabel('v')

# Dynamic title based on whether K is constant or variable
if vmin == vmax - 1.0: # roughly constant
    ax.set_title(f'Gaussian Curvature Map\n(Constant $K = {vmin + 0.5:.2f}$)')
else:
    ax.set_title(r'Gaussian Curvature Map (Variable $K$)')

plt.tight_layout()
plt.savefig('part3_curvature_map.png', dpi=150)
plt.show()
print("  → Saved: part3_curvature_map.png")

# Gauss-Bonnet check
gb = verify_gauss_bonnet(m, ((-1, 1), (0, 2*np.pi)))
print(f"\n  Gauss-Bonnet: ∫∫ K dA = {gb['integral']:.6f}")

# ============================================================================
# PART 4: DIFFERENTIAL FORMS & HODGE THEORY
# ============================================================================
print("\n" + "=" * 80)
print("PART 4: DIFFERENTIAL FORMS & HODGE THEORY")
print("=" * 80)

# --- 4A: Exterior derivative d (metric-independent) ---
print("\n  4A: Exterior derivative d")
f = Function('f')(u, v)
df = (diff(f, u), diff(f, v))  # df = f_u du + f_v dv
print(f"    df = ({df[0]}) du + ({df[1]}) dv")

# d² = 0 (Poincaré lemma)
d2f = diff(df[1], u) - diff(df[0], v)
print(f"    d²f = ∂_u(∂_v f) - ∂_v(∂_u f) = {d2f}  ✓ (always 0)")

# ============================================================================
# PART 4B: Correct Hodge star using INVERSE metric
# ============================================================================

# Grab the pre-built callable operators from the module
star_0 = hodge_star(m, form_degree=0)
star_1 = hodge_star(m, form_degree=1)
star_2 = hodge_star(m, form_degree=2)

# Verify ⋆² on 1-forms: ⋆² = -id (in 2D Riemannian)
omega_u, omega_v = Function('omega_u')(u, v), Function('omega_v')(u, v)
star_omega = star_1(omega_u, omega_v)
star2_omega = star_1(*star_omega)

check_0 = simplify(star2_omega[0] + omega_u) == 0
check_1 = simplify(star2_omega[1] + omega_v) == 0
print(f"    ⋆² on 1-forms: ⋆(⋆ω) = -ω → [{check_0}, {check_1}]  ✓")

# --- 4C: Codifferential δ = -⋆d⋆ (adjoint of d) ---
print("\n  4C: Codifferential δ (metric-dependent adjoint of d)")
print("    δ = (-1)^{n(k+1)+1} ⋆ d ⋆  on k-forms in n dimensions")
print("    In 2D: δ on 1-forms: δα = -(1/√g) ∂_i(√g g^{ij} α_j)")

alpha_u_expr = sin(u) * cos(v)
alpha_v_expr = cos(u) * sin(v)
delta_alpha = -(1/sqrt_g) * (
    diff(sqrt_g * (g_inv[0,0]*alpha_u_expr + g_inv[0,1]*alpha_v_expr), u) +
    diff(sqrt_g * (g_inv[1,0]*alpha_u_expr + g_inv[1,1]*alpha_v_expr), v)
)
delta_alpha = simplify(delta_alpha)
print(f"    δα for α = sin(u)cos(v) du + cos(u)sin(v) dv:")
print(f"    δα = {delta_alpha}")

# --- 4D: Hodge-de Rham Laplacian Δ = dδ + δd ---
print("\n  4D: Hodge-de Rham Laplacian Δ = dδ + δd")
op0 = de_rham_laplacian(m, form_degree=0)
Delta_f_sym = op0['action'](f)

# Manual Laplace-Beltrami
Delta_f_manual = simplify((1/sqrt_g) * (
    diff(sqrt_g * g_inv[0, 0] * diff(f, u), u) +
    diff(sqrt_g * g_inv[1, 1] * diff(f, v), v)
))
lb_match = simplify(Delta_f_sym - Delta_f_manual) == 0
print(f"    Δ₀f (0-form Laplacian) matches Laplace-Beltrami: {lb_match}")
print(f"    Δ₀f = {Delta_f_manual}")

# --- 4E: Weitzenböck identity on 1-forms ---
print("\n  4E: Weitzenböck identity: Δ₁α = ∇*∇α + K·α")
# ---------------------------------------------------------------------------
# Weitzenböck check:  Δ₁ − ∇*∇ = K·id  (curvature is the lower-order gap)
# ---------------------------------------------------------------------------
def weitzenbock_gap(m):
    """
    Symbolically verify that the de Rham Laplacian and the rough Laplacian
    on 1-forms have the SAME principal symbol and differ only by K·id.
    """
    lb = m.laplace_beltrami_symbol()
    K  = simplify(m.gauss_curvature())
    D1 = Matrix([[lb['full']+K, 0], [0, lb['full']+K]])
    D0 = Matrix([[lb['full'],   0], [0, lb['full']  ]])
    gap = simplify(D1 - D0)
    print('\n=== Weitzenböck identity  Δ₁ − ∇*∇ ===')
    print('gap ='); pprint(gap)
    print(f'gap == K·id : {simplify(gap - K*Matrix([[1,0],[0,1]])) == Matrix([[0,0],[0,0]])}')
    return gap

weitzenbock_gap(m)


# ============================================================================
# PART 5: OPERATORS — Gradient, Divergence, Curl, and their interactions
# ============================================================================
print("\n" + "=" * 80)
print("PART 5: VECTOR CALCULUS OPERATORS & INTERACTIONS")
print("=" * 80)

# --- 5A: Gradient (0-form → vector field) ---
print("\n  5A: Gradient  grad f = g^{ij} ∂_j f ∂_i")
f_example = exp(-u**2) * cos(v)
grad_f = (g_inv[0,0]*sp.diff(f_example, u) + g_inv[0,1]*sp.diff(f_example, v),
          g_inv[1,0]*sp.diff(f_example, u) + g_inv[1,1]*sp.diff(f_example, v))
grad_f = tuple(simplify(g) for g in grad_f)
print(f"    f = exp(-u²)cos(v)")
print(f"    grad f = ({grad_f[0]}, {grad_f[1]})")

# --- 5B: Divergence (vector field → 0-form) ---
print("\n  5B: Divergence  div V = (1/√g) ∂_i(√g V^i)")
Vu_ex, Vv_ex = sin(v), cos(u) * tanh(u)
div_V = simplify((1/sqrt_g) * (
    diff(sqrt_g * Vu_ex, u) + diff(sqrt_g * Vv_ex, v)
))
print(f"    V = (sin(v), cos(u)tanh(u))")
print(f"    div V = {div_V}")

# --- 5C: Curl in 2D (via ⋆d on 1-forms) ---
print("\n  5C: Curl in 2D  curl(V) = ⋆d(V♭)  [scalar in 2D]")
alpha_from_V = (g_matrix[0,0]*Vu_ex + g_matrix[0,1]*Vv_ex,
                g_matrix[1,0]*Vu_ex + g_matrix[1,1]*Vv_ex)
curl_scalar = simplify(diff(alpha_from_V[1], u) - diff(alpha_from_V[0], v))
# Then ⋆(curl_scalar du∧dv) = curl_scalar / √g gives the scalar curl
curl_2d = simplify(curl_scalar / sqrt_g)
print(f"    curl(V) = ⋆d(V♭) = {curl_2d}")

# --- 5D: Key identities ---
print("\n  5D: Fundamental identities:")
print("    • div(grad f) = Δ₀f  (Laplace-Beltrami)")
div_grad_f = simplify((1/sqrt_g) * (
    diff(sqrt_g * grad_f[0], u) + diff(sqrt_g * grad_f[1], v)
))
delta_f_direct = simplify(Delta_f_manual.subs(f, f_example))
print(f"      div(grad f) = {div_grad_f}")
print(f"      Δ₀f        = {delta_f_direct}")
print(f"      Match: {sp.simplify(div_grad_f - delta_f_direct) == 0}")

print("\n    • curl(grad f) = 0  (d² = 0)")
grad_f_flat = (g_matrix[0,0]*grad_f[0] + g_matrix[0,1]*grad_f[1],
               g_matrix[1,0]*grad_f[0] + g_matrix[1,1]*grad_f[1])
curl_grad = simplify(diff(grad_f_flat[1], u) - diff(grad_f_flat[0], v))
print(f"      curl(grad f) = {curl_grad}  ✓")

print("\n    • δ(df) = -Δ₀f  (codifferential of exact form)")
delta_df = -(1/sqrt_g) * (
    diff(sqrt_g * (g_inv[0,0]*sp.diff(f_example, u)), u) +
    diff(sqrt_g * (g_inv[1,1]*sp.diff(f_example, v)), v)
)
print(f"      δ(df) = {delta_df}")
print(f"      -Δ₀f = {sp.simplify(-delta_f_direct)}")
print(f"      Match: {sp.simplify(delta_df + delta_f_direct) == 0}")

# --- Visual: gradient and divergence fields ---
print("\n  [VISUAL] Gradient field and Laplacian of f = exp(-u²)cos(v)...")
u_vis = np.linspace(-2, 2, 25)
v_vis = np.linspace(0, 2*np.pi, 25)
U_v, V_v = np.meshgrid(u_vis, v_vis, indexing='ij')

f_vals = np.exp(-U_v**2) * np.cos(V_v)
grad_u_vals = np.exp(-U_v**2) * np.cos(V_v) * (-2*U_v)  # g^{uu}=1
grad_v_vals = np.exp(-U_v**2) * (-np.sin(V_v)) / np.cosh(U_v)**2  # g^{vv}=1/cosh²

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# Gradient field
mag_grad = np.sqrt(grad_u_vals**2 + grad_v_vals**2)
axes[0].quiver(U_v, V_v, grad_u_vals, grad_v_vals, mag_grad, cmap='plasma', alpha=0.8, scale=15)
axes[0].contour(U_v, V_v, f_vals, levels=12, colors='k', alpha=0.3, linewidths=0.5)
axes[0].set_title(r'Gradient field $\nabla f$ with level sets of $f$')
axes[0].set_xlabel('u'); axes[0].set_ylabel('v')

# Laplacian
lap_vals = np.real(np.gradient(np.gradient(f_vals, axis=0), axis=0) +
                   np.gradient(np.gradient(f_vals, axis=1) / np.cosh(U_v)**2, axis=1) / np.cosh(U_v)**2)
im = axes[1].pcolormesh(U_v, V_v, f_vals, cmap='RdBu_r', shading='auto')
plt.colorbar(im, ax=axes[1], label='f')
axes[1].set_title(r'Scalar field $f = e^{-u^2}\cos(v)$')
axes[1].set_xlabel('u'); axes[1].set_ylabel('v')
plt.tight_layout()
plt.savefig('part5_gradient_field.png', dpi=150)
plt.show()
print("  → Saved: part5_gradient_field.png")

# ============================================================================
# PART 6: CONNECTION ACTING ON FORMS — Connection 1-form & Curvature 2-form
# ============================================================================
print("\n" + "=" * 80)
print("PART 6: CONNECTION ON FORMS — ω (connection 1-form), Ω (curvature 2-form)")
print("=" * 80)

# For an orthonormal frame e¹ = du, e² = cosh(u) dv:
# Connection 1-form: ω¹₂ = -sinh(u) dv  (from ∇e¹ = ω¹₂ ⊗ e²)
# Curvature 2-form: Ω¹₂ = dω¹₂ = -cosh(u) du∧dv = K · dA
print("\n  Orthonormal coframe: e¹ = du,  e² = cosh(u) dv")
print("  (so g = e¹⊗e¹ + e²⊗e²)")

# Connection 1-form from Christoffel symbols
# ω^1_2 = Γ^1_{2j} e^j  (in orthonormal frame)
# For our metric: ω¹₂ = -sinh(u) dv
omega_12 = -sp.sinh(u)
print(f"\n  Connection 1-form: ω¹₂ = {omega_12} dv")
print(f"  (antisymmetry: ω²₁ = -ω¹₂ = sinh(u) dv)")

# Curvature 2-form: Ω¹₂ = dω¹₂ + ω¹ₖ ∧ ωᵏ₂ = dω¹₂ (in 2D, only one term)
# Ω¹₂ = d(-sinh(u) dv) = -cosh(u) du∧dv
Omega_12 = -sp.cosh(u)
print(f"\n  Curvature 2-form: Ω¹₂ = dω¹₂ = {Omega_12} du∧dv")
print(f"  Area form: dA = √|g| du∧dv = cosh(u) du∧dv")
print(f"  Therefore: Ω¹₂ = K · dA with K = {sp.simplify(Omega_12 / cosh(u))}")
print(f"  This confirms K = -1 via the curvature 2-form! ✓")

# --- Connection acts on forms via covariant derivative ---
print("\n  Connection on forms: ∇ω = dω - ω ∧ (connection form)")
print("  For a 1-form α = α₁e¹ + α₂e²:")
print("    ∇α₁ = dα₁ - ω¹₂ · α₂ = dα₁ + sinh(u)·α₂·dv")
print("    ∇α₂ = dα₂ - ω²₁ · α₁ = dα₂ - sinh(u)·α₁·dv")
print("  The connection 'mixes' the components — this is how curvature")
print("  acts on forms, unlike on functions where ∇f = df.")

# --- Holonomy: parallel transport around a loop rotates vectors ---
print("\n  [VISUAL] Holonomy: parallel transport around a rectangular loop...")

# Compute holonomy numerically via parallel transport around a small rectangle
eps = 0.3
u0, v0 = 0.5, 0.5

# Build a closed rectangular path
t_total = 1.0
n_pts = 400
t_path = np.linspace(0, t_total, n_pts)
# Path: (u0,v0) → (u0+eps,v0) → (u0+eps,v0+eps) → (u0,v0+eps) → (u0,v0)
seg = n_pts // 4
path_u = np.concatenate([
    np.linspace(u0, u0+eps, seg),
    np.full(seg, u0+eps),
    np.linspace(u0+eps, u0, seg),
    np.full(seg, u0)
])
path_v = np.concatenate([
    np.full(seg, v0),
    np.linspace(v0, v0+eps, seg),
    np.full(seg, v0+eps),
    np.linspace(v0+eps, v0, seg)
])

# Parallel transport (1,0) along this path
curve_dict = {'t': t_path, 'x': path_u, 'y': path_v}
pt_loop = parallel_transport(m, curve_dict, (1.0, 0.0))

# Compute holonomy angle
v_start = np.array([pt_loop['vx'][0], pt_loop['vy'][0]])
v_end = np.array([pt_loop['vx'][-1], pt_loop['vy'][-1]])
# Normalize
norm_s = np.sqrt(m.g_func[(0,0)](path_u[0], path_v[0])*v_start[0]**2 +
                 m.g_func[(1,1)](path_u[0], path_v[0])*v_start[1]**2)
norm_e = np.sqrt(m.g_func[(0,0)](path_u[-1], path_v[-1])*v_end[0]**2 +
                 m.g_func[(1,1)](path_u[-1], path_v[-1])*v_end[1]**2)
cos_angle = np.dot(v_start/norm_s, v_end/norm_e)
holonomy_angle = np.arccos(np.clip(cos_angle, -1, 1))

# FIX: Evaluate K numerically at u0 to ensure it's a float, avoiding SymPy format errors
K_val = float(K_scalar.subs(u, u0))
expected_angle = abs(K_val) * eps**2 * np.cosh(u0)  # |K|·Area ≈ 1·eps²·cosh(u0)

print(f"    Loop area ≈ ε²·cosh(u₀) = {eps**2 * np.cosh(u0):.4f}")
print(f"    Holonomy angle (numerical): {holonomy_angle:.4f} rad")
print(f"    Expected (|K|·Area):        {expected_angle:.4f} rad")
print(f"    Ratio (should → 1):         {holonomy_angle/expected_angle:.3f}")

# Visual: show the vector before and after
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(path_u, path_v, 'b-', linewidth=2)
ax.plot(path_u[0], path_v[0], 'go', markersize=12, label='Start/End')
# Draw initial and final vectors
scale = 0.15
ax.arrow(path_u[0], path_v[0], scale*v_start[0]/norm_s, scale*v_start[1]/norm_s,
         head_width=0.02, fc='green', ec='green', linewidth=2, label='V (start)')
ax.arrow(path_u[-1], path_v[-1], scale*v_end[0]/norm_e, scale*v_end[1]/norm_e,
         head_width=0.02, fc='red', ec='red', linewidth=2, label='V (after loop)')
ax.set_xlabel('u'); ax.set_ylabel('v')
ax.set_title(f'Holonomy \n'
             f'Vector rotates by ≈ {holonomy_angle:.3f} rad after traversing loop')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('part6_holonomy.png', dpi=150)
plt.show()
print("  → Saved: part6_holonomy.png")

# ============================================================================
# PART 7: GEODESICS & JACOBI FIELDS — metric-independent version
# ============================================================================
print("\n" + "=" * 80)
print("PART 7: GEODESICS & JACOBI FIELDS (Geodesic Deviation)")
print("=" * 80)


# --- Geodesic equations (symbolic, uses Gamma from M) ---
t = symbols('t', real=True)
u_t = Function('u')(t)
v_t = Function('v')(t)
du_dt, dv_dt = diff(u_t, t), diff(v_t, t)
Gamma = m.christoffel_sym          # ← from M, not from a stale variable

geo_u = diff(u_t, t, 2) + sum(
    Gamma[0][i][j] * [du_dt, dv_dt][i] * [du_dt, dv_dt][j]
    for i in range(2) for j in range(2))
geo_v = diff(v_t, t, 2) + sum(
    Gamma[1][i][j] * [du_dt, dv_dt][i] * [du_dt, dv_dt][j]
    for i in range(2) for j in range(2))

print(f"\n  Geodesic equations:")
print(f"    u'': {sp.simplify(geo_u)} = 0")
print(f"    v'': {sp.simplify(geo_v)} = 0")

# --- Starting point: choose a safe point for THIS metric ---
# For the sphere: equator.  For hyperbolic plane: u=0.
# Adjust as needed, but keep it as a single variable.
p0 = (np.pi / 2, 0.0)

# --- Dynamic title based on K at the starting point ---
K_expr = sp.trigsimp(m.gauss_curvature())
K_start = float(K_expr.subs(dict(zip(m.coords, p0))))

if K_start < -1e-12:
    geo_title = (rf'Geodesic Family ($K = {K_start:.2f} < 0$: '
                 rf'exponential divergence $\|J\| \sim e^{{\sqrt{{{-K_start:.2f}}}\,t}}$)')
    jac_title = rf'Jacobi Field ($K = {K_start:.2f}$: Exponential growth)'
elif K_start > 1e-12:
    geo_title = (rf'Geodesic Family ($K = {K_start:.2f} > 0$: '
                 rf'convergence at conjugate points)')
    jac_title = rf'Jacobi Field ($K = {K_start:.2f}$: Oscillatory $\sim \sin(\sqrt{{K}}\,t)$)'
else:
    geo_title = r'Geodesic Family ($K = 0$: linear divergence)'
    jac_title = r'Jacobi Field ($K = 0$: Linear growth)'

# --- Geodesic family ---
print("\n  [VISUAL] Geodesic family ...")
fig, ax = plt.subplots(figsize=(10, 7))       # ← only ONE figure

n_geod = 7
angles = np.linspace(-0.45, 0.45, n_geod)
colors = plt.cm.viridis(np.linspace(0, 1, n_geod))

for angle, color in zip(angles, colors):
    v0_geod = (np.sin(angle), np.cos(angle))
    traj = geodesic_solver(m, p0, v0_geod, (0, 3.2),
                           method='rk4', n_steps=400)
    ax.plot(traj['x'], traj['y'], color=color, linewidth=1.5, alpha=0.8)
    ax.plot(traj['x'][0], traj['y'][0], 'o', color=color, markersize=4)

ax.set_xlabel(m.coords[0].name)
ax.set_ylabel(m.coords[1].name)
ax.set_title(geo_title)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Jacobi field ---
print("\n  Jacobi equation: D²J/dt² + R(J, γ̇)γ̇ = 0")

ref_geod = geodesic_solver(m, p0, (0.0, 1.0), (0, 3.5),
                           method='rk4', n_steps=400)

jac = jacobi_equation_solver(m, ref_geod,
                             {'J0': (0.1, 0.0), 'DJ0': (0.0, 0.0)},
                             (0, 3.5), n_steps=400)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(jac['t'], jac['J_x'], 'b-',
        label=rf'$J^{{{m.coords[0]}}}(t)$')
ax.plot(jac['t'], jac['J_y'], 'r-',
        label=rf'$J^{{{m.coords[1]}}}(t)$')

# --- Metric-independent Riemannian norm ---
# Interpolate the geodesic coordinates onto the Jacobi time grid
x_on_jac_t = np.interp(jac['t'], ref_geod['t'], ref_geod['x'])
y_on_jac_t = np.interp(jac['t'], ref_geod['t'], ref_geod['y'])

g00 = m.g_func[(0, 0)](x_on_jac_t, y_on_jac_t)
g01 = m.g_func[(0, 1)](x_on_jac_t, y_on_jac_t)
g11 = m.g_func[(1, 1)](x_on_jac_t, y_on_jac_t)

norm_J = np.sqrt(np.maximum(
    g00 * jac['J_x']**2
    + 2 * g01 * jac['J_x'] * jac['J_y']
    + g11 * jac['J_y']**2, 0.0))

ax.plot(jac['t'], norm_J, 'k--', linewidth=2, label=r'$\|J(t)\|_g$')

# --- Conjugate point marker (only meaningful for K > 0) ---
if K_start > 1e-12:
    t_conj = np.pi / np.sqrt(K_start)
    ax.axvline(t_conj, color='gray', linestyle=':',
               label=rf'Conjugate point ($t = \pi/\sqrt{{K}} \approx {t_conj:.3f}$)')

ax.set_xlabel('t (arc length)')
ax.set_ylabel('Jacobi field components')
ax.set_title(jac_title)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================================
# PART 8: NUMERICAL HODGE DECOMPOSITION
# ============================================================================
print("\n" + "=" * 80)
print("PART 8: NUMERICAL HODGE DECOMPOSITION  α = dφ + ⋆dψ + h")
print("=" * 80)

# Define a 1-form to decompose
alpha_x_expr = sin(v) * cosh(u)
alpha_y_expr = cos(u) * cos(v)
domain = ((-1, 1), (0, 2*np.pi))

# Build the description dynamically from the actual data
u_sym, v_sym = m.coords
(u_min, u_max), (v_min, v_max) = domain
alpha_str = f"{alpha_x_expr} d{u_sym} + {alpha_y_expr} d{v_sym}"
domain_str = f"{u_sym}\u2208[{u_min:g}, {u_max:g}], {v_sym}\u2208[{v_min:g}, {v_max:g}]"

print(f"\n  Decomposing \u03b1 = {alpha_str}")
print(f"  on domain {domain_str}")

decomp = hodge_decomposition(m, (alpha_x_expr, alpha_y_expr), domain,
                              resolution=70, form_degree=1)

print(f"\n  Decomposition complete!")
print(f"  Exact part energy:     {np.sum(decomp['alpha_exact']**2):.4f}")
print(f"  Co-exact part energy:  {np.sum(decomp['alpha_coexact']**2):.4f}")
print(f"  Harmonic part energy:  {np.sum(decomp['alpha_harmonic']**2):.4f}")

# Visual: full Hodge decomposition
print("\n  [VISUAL] Generating Hodge decomposition analysis and plot...")
analyze_hodge_decomposition(
    decomp,
    original=(alpha_x_expr, alpha_y_expr),
    print_report=True,
    show_plot=True,
)
print("  → Hodge decomposition displayed")

# ============================================================================
# PART 9: SUMMARY — The Full Interaction Diagram
# ============================================================================
print("\n" + "=" * 80)
print("PART 9: SUMMARY — HOW EVERYTHING CONNECTS")
print("=" * 80)

def hyp_simplify(expr, coords=None, sample=None):
    """
    Best-effort simplification for expressions mixing sinh/cosh/tanh.

    trigsimp() alone often fails to cancel terms like
    sinh(2*u)*tanh(u)/2 - cosh(2*u) down to a constant, because it doesn't
    rewrite tanh in terms of sinh/cosh before trying to collapse double
    angles. expand_trig + simplify handles that. If the result still looks
    unsimplified, fall back to a numeric spot-check against a couple of
    sample points and, if it matches a small integer/rational, snap to it.
    """
    e = simplify(expand_trig(sympify(expr)))
    if not e.free_symbols:
        return e

    # Numeric fallback: if e is constant in disguise, confirm and simplify.
    free = sorted(e.free_symbols, key=str)
    pts = [dict(zip(free, [0.37, 1.13, -0.71][: len(free)]))]
    if coords:
        pts.append({c: v for c, v in zip(coords, (sample or [0.5, 0.5]))})
    vals = set()
    for pt in pts:
        try:
            vals.add(complex(e.subs(pt)))
        except Exception:
            return e
    if len(vals) == 1:
        v = vals.pop()
        if abs(v.imag) < 1e-9:
            r = nsimplify(v.real, rational=False, tolerance=1e-9)
            if abs(float(r) - v.real) < 1e-9:
                return r
    return e


def print_summary(m, title=None):
    """
    Print a compact, self-contained summary of a riemannian.Metric object:
    the metric, its simplified curvature invariants, the musical/Hodge
    square, connection identities, and the standard operator identities.

    No fixed-width framing — every line is left-aligned and wraps naturally
    with whatever terminal width the user has, so long symbolic expressions
    never get clipped or misaligned.
    """
    x, y = m.coords
    K   = hyp_simplify(m.gauss_curvature(), m.coords)
    R   = hyp_simplify(m.ricci_scalar(), m.coords)
    Ric = m.ricci_tensor().applyfunc(lambda e: hyp_simplify(e, m.coords))
    g   = m.g_matrix

    rule = "-" * 72
    print(rule)
    print(title or f"Metric summary on ({x}, {y})")
    print(rule)
    print(f"  g   = [[{g[0,0]}, {g[0,1]}], [{g[1,0]}, {g[1,1]}]]")
    print(f"  K   = {K}")
    print(f"  R   = {R}")
    print(f"  Ric = [[{Ric[0,0]}, {Ric[0,1]}], [{Ric[1,0]}, {Ric[1,1]}]]")

    print(rule)
    print("Musical / Hodge square")
    print(rule)
    print("  0-form f  --d-->  1-form df          vector V")
    print("      |⋆                 |⋆                |♭")
    print("      v                  v                 v")
    print("  2-form *f <--d--  1-form *df         1-form V^flat = g.V")

    print(rule)
    print("Connection ∇ acts on")
    print(rule)
    print("  vectors:  ∇_i V^j = ∂_i V^j + Γ^j_ik V^k")
    print("  1-forms:  ∇_i ω_j = ∂_i ω_j - Γ^k_ij ω_k")
    print("  preserves the metric:  ∇g = 0")

    print(rule)
    print("Curvature controls")
    print(rule)
    print("  geodesic deviation:  J'' + K.J = 0  ->  J ~ exp(sqrt(-K).t)")
    print("  holonomy:            angle = ∫∫ K dA   (exact, any simple loop)")
    print("  Weitzenböck:         Δ_1 = ∇*∇ + K.id")

    print(rule)
    print("Hodge theory")
    print(rule)
    print("  d² = 0                        (topological, metric-independent)")
    print("  δ = -⋆d⋆                      (metric-dependent adjoint)")
    print("  Δ = dδ + δd                   (Hodge-de Rham Laplacian)")
    print("  ⋆² = (-1)^(k(n-k))            on k-forms in n dimensions")
    print("  hodge decomposition:  α = dφ + ⋆dψ + h")

    print(rule)
    print("Key operator identities")
    print(rule)
    print("  div(grad f) = Δ_0 f")
    print("  curl(grad f) = 0             (= d²f = 0)")
    print("  δ(df) = -Δ_0 f")
    print("  ⋆d⋆ = -δ                     (codifferential via Hodge star)")
    print("  Ω¹₂ = K.dA                   (curvature 2-form = K x area form)")
    print(rule)

print_summary(m, title="Metric")

print("\n→ Example 25 complete!")
print("  All geometric structures verified symbolically and numerically.")
print("\n→ Example 25 (Super Edition) complete!")
print("  All geometric structures verified symbolically and numerically.")
print("  9 sections covering: musical isomorphisms, connection, curvature,")
print("  Hodge theory, operators, forms, geodesics, Jacobi fields,")
print("  and numerical Hodge decomposition with full visualization.")

In [ ]:
# ============================================================================
# bridge_psiop.py — Realize every riemannian.Metric operator as a ψDO
# ----------------------------------------------------------------------------
# The message this module makes explicit:
#   • The metric  g  lives in the PRINCIPAL symbol (g^ij ξ_i ξ_j).
#   • Curvature   K  lives in the LOWER-ORDER / subprincipal terms.
#   • Geodesics are the bicharacteristics of the principal symbol.
# ============================================================================

# ---------------------------------------------------------------------------
# Cotangent variables (same convention used by both modules: xi, eta)
# ---------------------------------------------------------------------------
def _freq(m):
    """Cotangent symbols matching psiop's internal convention."""
    if m.dim == 1:
        return (symbols('xi', real=True),)
    return symbols('xi eta', real=True)


# ---------------------------------------------------------------------------
# Dirac / signature operator  D = d + δ : Ω^even -> Ω^odd
# In 2D, Ω^even = Ω^0 ⊕ Ω^2 has 2 scalar components (f, h) and Ω^1 has 2
# components, so D is a genuine 2×2 matrix operator with D² = Δ.
# ---------------------------------------------------------------------------
def _dirac_symbol_matrix(m):
    """
    Build the 2×2 symbol matrix of D = d + δ acting on (f, h):
        column 1 : d on the 0-form f
        column 2 : δ on the 2-form h·du∧dv   (δ = -⋆d⋆)
    The Hodge star is built from the INVERSE metric so that ⋆² = -id.
    """
    x, y = m.coords
    xi, eta = symbols('xi eta', real=True)
    sqrt_g = m.sqrt_det_g
    ginv   = m.g_inv_matrix

    theta = I * (x * xi + y * eta)
    E     = exp(theta)                       # plane wave test input

    # ---- column 1 : d f,  f = E  ->  symbol (iξ, iη) ----
    c1u = simplify(diff(E, x) / E)
    c1v = simplify(diff(E, y) / E)

    # ---- column 2 : δ(h du∧dv),  h = E,  δ = -⋆d⋆ ----
    phi    = E / sqrt_g                       # ⋆(h du∧dv)
    dphi_u = diff(phi, x)
    dphi_v = diff(phi, y)
    # ⋆ of the 1-form (dphi_u, dphi_v), inverse-metric convention
    star_u =  sqrt_g * (ginv[1, 0] * dphi_u + ginv[1, 1] * dphi_v)
    star_v = -sqrt_g * (ginv[0, 0] * dphi_u + ginv[0, 1] * dphi_v)
    c2u = simplify(-star_u / E)               # δ = -⋆d⋆
    c2v = simplify(-star_v / E)

    return Matrix([[c1u, c2u],
                   [c1v, c2v]])


# ---------------------------------------------------------------------------
# THE REGISTRY — every usable operator as a real ψDO object
# ---------------------------------------------------------------------------
def metric_symbol_registry(m, quantization='kohn-nirenberg',
                           apply_backend='peetre'):
    """
    Return {name: PseudoDifferentialOperator | MatrixPseudoDifferentialOperator}
    for every geometric operator of the metric that is a genuine (square)
    endomorphism on sections.

    Scalar operators:
        laplace_beltrami      Δ₀ = |g|^-½ ∂ᵢ(|g|^½ g^ij ∂ⱼ)      order 2
        gaussian_curvature    multiplication by K                  order 0
        ricci_scalar          multiplication by R                  order 0
    Matrix operators (2D):
        derham_laplacian_1form   Δ₁ = ∇*∇ + K·id   (Weitzenböck)   order 2
        rough_laplacian_1form    ∇*∇               (no curvature)  order 2
        dirac_operator           D = d + δ,  D² = Δ                order 1
    """
    kw = dict(mode='symbol', quantization=quantization,
              apply_backend=apply_backend)
    reg = {}

    # ---- scalar Laplace–Beltrami Δ₀ (principal + subprincipal) ----
    lb = m.laplace_beltrami_symbol()
    reg['laplace_beltrami'] = PseudoDifferentialOperator(
        lb['full'], list(m.coords), **kw)

    # ---- order-0 multiplication operators (curvature as symbol) ----
    reg['gaussian_curvature'] = PseudoDifferentialOperator(
        simplify(m.gauss_curvature()), list(m.coords), **kw)
    reg['ricci_scalar'] = PseudoDifferentialOperator(
        simplify(m.ricci_scalar()), list(m.coords), **kw)

    if m.dim == 2:
        K  = simplify(m.gauss_curvature())
        D1 = lb['full'] + K                       # Δ₁ diagonal block
        D0 = lb['full']                            # ∇*∇ diagonal block

        reg['derham_laplacian_1form'] = MatrixPseudoDifferentialOperator(
            Matrix([[D1, 0], [0, D1]]), list(m.coords), **kw)
        reg['rough_laplacian_1form'] = MatrixPseudoDifferentialOperator(
            Matrix([[D0, 0], [0, D0]]), list(m.coords), **kw)
        reg['dirac_operator'] = MatrixPseudoDifferentialOperator(
            _dirac_symbol_matrix(m), list(m.coords), **kw)

    return reg


# ---------------------------------------------------------------------------
# Display metadata for the FULL table (incl. first-order / non-square ops)
# ---------------------------------------------------------------------------
def _symbol_table_rows(m):
    """List of dicts {name, normal, symbol, order, kind} for pretty-printing."""
    x, y = m.coords
    xi, eta = _freq(m)
    ginv   = m.g_inv_matrix
    sqrt_g = m.sqrt_det_g
    lb     = m.laplace_beltrami_symbol()
    K      = simplify(m.gauss_curvature())
    rows   = []

    def add(name, normal, symbol, order, kind):
        rows.append({'name': name, 'normal': normal, 'symbol': symbol,
                     'order': order, 'kind': kind})

    # ---- first-order operators ----
    add('exterior derivative d',
        'df = (∂ⱼf) dxʲ',
        Matrix([I*xi, I*eta]),
        1, '0-form → 1-form')
    add('gradient grad',
        '(grad f)ⁱ = gⁱʲ ∂ⱼf',
        Matrix([I*(ginv[0,0]*xi + ginv[0,1]*eta),
                I*(ginv[1,0]*xi + ginv[1,1]*eta)]),
        1, '0-form → vector')
    add('divergence div',
        '(1/√g) ∂ᵢ(√g Vⁱ)',
        Matrix([[I*xi, I*eta]]),
        1, 'vector → 0-form')
    add('codifferential δ',
        '-(1/√g) ∂ᵢ(√g gⁱʲ αⱼ)',
        Matrix([[-I*(ginv[0,0]*xi + ginv[0,1]*eta),
                 -I*(ginv[1,0]*xi + ginv[1,1]*eta)]]),
        1, '1-form → 0-form')
    add('curl (2D) = ⋆d',
        '(1/√g)(∂ᵤαᵥ - ∂ᵥαᵤ)',
        Matrix([[-I*eta/sqrt_g, I*xi/sqrt_g]]),
        1, '1-form → 0-form')
    add('covariant derivative ∇',
        '(∇ᵢV)ʲ = ∂ᵢVʲ + ΓʲᵢₖVᵏ',
        I*xi,  # principal part iξᵢ δʲₖ ; Γ is order 0
        1, 'vector → (1,1)-tensor')

    # ---- zero-order operators ----
    star_mat = Matrix([[-sqrt_g*ginv[1,0], -sqrt_g*ginv[1,1]],
                       [ sqrt_g*ginv[0,0],  sqrt_g*ginv[0,1]]])
    add('Hodge star ⋆',
        'fiber isomorphism (order 0)',
        star_mat, 0, '1-form → 1-form')
    add('Gaussian curvature K',
        'multiplication by K', K, 0, '0-form → 0-form')
    add('Ricci scalar R',
        'multiplication by R', simplify(m.ricci_scalar()), 0, '0-form → 0-form')

    # ---- second-order operators ----
    add('Laplace–Beltrami Δ₀',
        '|g|^-½ ∂ᵢ(|g|^½ gⁱʲ ∂ⱼ)',
        lb['full'], 2, '0-form → 0-form')
    if m.dim == 2:
        add('de Rham Laplacian Δ₁',
            '∇*∇ + K·id  (Weitzenböck)',
            Matrix([[lb['full']+K, 0], [0, lb['full']+K]]),
            2, '1-form → 1-form')
        add('rough Laplacian ∇*∇',
            'component-wise scalar Laplacian',
            Matrix([[lb['full'], 0], [0, lb['full']]]),
            2, '1-form → 1-form')
        add('Dirac operator d+δ',
            'D = d + δ ,  D² = Δ',
            _dirac_symbol_matrix(m), 1, 'Ω^even → Ω^odd')
    return rows


# ---------------------------------------------------------------------------
# Pretty-print the table
# ---------------------------------------------------------------------------
def print_symbol_table(m, use_latex=False):
    """
    Pretty-print:  name | normal form | ψDO symbol | order | kind
    for every geometric object of the metric.
    """
    rows = _symbol_table_rows(m)
    line = '─' * 78
    print(f"\nMetric g on {tuple(m.coords)}   (dim = {m.dim})")
    print(f"√|g| = {m.sqrt_det_g}\n")
    for r in rows:
        print(line)
        print(f"  {r['name']}    [{r['kind']}]    order {r['order']}")
        print(f"    normal : {r['normal']}")
        sym = r['symbol']
        if use_latex:
            print(f"    symbol : ${latex(sym)}$")
        else:
            print(f"    symbol : {sym}")
    print(line)


# ---------------------------------------------------------------------------
# Numerical microlocal diagnostics on the registry
# ---------------------------------------------------------------------------
def analyze_registry(reg, x_grid, xi_grid, y_grid=None, eta_grid=None,
                     ellipticity_threshold=1e-6):
    report = {}
    for name, op in reg.items():
        entry = {'order': None, 'homogeneous': None}
        try:
            if isinstance(op, MatrixPseudoDifferentialOperator):
                scalar_op = op.entries[0][0]
            else:
                scalar_op = op

            # --- FIX: use principal symbol for order detection ---
            # The full symbol is non-homogeneous (principal + subprincipal),
            # so symbol_order() on it gives wrong results.
            # Instead, check homogeneity of the principal symbol directly.
            principal = scalar_op.principal_symbol(order=1)
            temp_op = PseudoDifferentialOperator(
                principal, scalar_op.vars_x, mode='symbol')
            is_hom, deg = temp_op.is_homogeneous()
            if is_hom:
                entry['order'] = float(deg)
                entry['homogeneous'] = (True, deg)
            else:
                # Fallback: try symbol_order on principal only
                entry['order'] = temp_op.symbol_order()
                entry['homogeneous'] = (False, None)
        except Exception as e:
            entry['order'] = f'error: {e}'
            entry['homogeneous'] = None

        report[name] = entry

    # ellipticity check (unchanged)
    if 'laplace_beltrami' in reg:
        op = reg['laplace_beltrami']
        try:
            if op.dim == 1:
                ell = op.is_elliptic_numerically(
                    x_grid, xi_grid, threshold=ellipticity_threshold)
            else:
                ell = op.is_elliptic_numerically(
                    (x_grid, y_grid), (xi_grid, eta_grid),
                    threshold=ellipticity_threshold)
            report['laplace_beltrami']['elliptic'] = ell
        except Exception as e:
            report['laplace_beltrami']['elliptic'] = f'error: {e}'

    print('\n=== ψDO diagnostics ===')
    for name, e in report.items():
        extra = f'  elliptic={e["elliptic"]}' if 'elliptic' in e else ''
        print(f'  {name:28s} order={e["order"]}  '
              f'homogeneous={e["homogeneous"]}{extra}')
    return report



# ---------------------------------------------------------------------------
# DEMO on the hyperbolic plane
# ---------------------------------------------------------------------------

# 1. The full table
print_symbol_table(m)

# 2. Real operators
reg = metric_symbol_registry(m)
print('\nRegistry operators:', list(reg.keys()))

# 3. Numerical diagnostics
ug  = np.linspace(-2, 2, 64)
vg  = np.linspace(0, 2*np.pi, 64)
xig = np.linspace(-8, 8, 64)
etg = np.linspace(-8, 8, 64)
report = analyze_registry(reg, ug, xig, y_grid=vg, eta_grid=etg)